In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import segmentation_models_pytorch as smp
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from tqdm import tqdm

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask_arr = np.array(mask) if not isinstance(mask, np.ndarray) else mask
    unique_values = np.unique(mask_arr)
    remap_dict = {val: idx for idx, val in enumerate(sorted(unique_values))}
    remapped_mask = np.zeros_like(mask_arr)

    for original_val, new_val in remap_dict.items():
        remapped_mask[mask_arr == original_val] = new_val

    return remapped_mask

In [ ]:
#TO DO

class customDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None, img_size=(256, 256)):
        self.root_dir = root_dir
        self.images_dir = os.path.join(root_dir, "dataset", "images")
        self.masks_dir = os.path.join(root_dir, "dataset", "masks")
        self.transform = transform
        self.img_size = img_size
        self.image_files = sorted([f for f in os.listdir(self.images_dir) if f.endswith('.jpg')])


    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert('RGB')


        mask_name = img_name.replace('.jpg', '.png')
        mask_path = os.path.join(self.masks_dir, mask_name)
        mask = Image.open(mask_path).convert('L')

        mask = remap_mask(mask)
        mask_pil = Image.fromarray(mask.astype(np.uint8))
        mask_pil = mask_pil.resize((self.img_size[1], self.img_size[0]), Image.NEAREST)
        mask = np.array(mask_pil)

        if self.transform:
            image = self.transform['image'](image)

        mask = torch.from_numpy(mask).long()
        return image, mask

In [ ]:
import os
import glob
from PIL import Image
import numpy as np

print("Dataset structure:")
print(os.listdir(path))

# Update path to include dataset subdirectory
dataset_path = os.path.join(path, "dataset")
print(f"\nDataset subdirectory: {os.listdir(dataset_path)}")

images_dir = os.path.join(dataset_path, "images")
masks_dir = os.path.join(dataset_path, "masks")

print(f"\nImages directory exists: {os.path.exists(images_dir)}")
print(f"Masks directory exists: {os.path.exists(masks_dir)}")

if os.path.exists(images_dir):
    sample_images = os.listdir(images_dir)[:3]
    print(f"\nSample images: {sample_images}")
    print(f"Total images: {len(os.listdir(images_dir))}")

    sample_img_path = os.path.join(images_dir, sample_images[0])
    img = Image.open(sample_img_path)
    print(f"Sample image size: {img.size}, mode: {img.mode}")

if os.path.exists(masks_dir):
    sample_masks = os.listdir(masks_dir)[:3]
    print(f"\nSample masks: {sample_masks}")
    print(f"Total masks: {len(os.listdir(masks_dir))}")

    sample_mask_path = os.path.join(masks_dir, sample_masks[0])
    mask = Image.open(sample_mask_path)
    print(f"Sample mask size: {mask.size}, mode: {mask.mode}")

    mask_arr = np.array(mask)
    print(f"Unique mask values: {np.unique(mask_arr)}")

In [ ]:
transform = {
    'image': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
}

dataset = customDataset(path, transform=transform)
image, mask = dataset[0]
print(f"Dataset size: {len(dataset)}")
print(f"Image: {image.shape}, Mask: {mask.shape}")
print(f"Classes: {len(torch.unique(mask))}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_vis = torch.clamp(image * std + mean, 0, 1)

axes[0].imshow(img_vis.permute(1, 2, 0))
axes[0].set_title("Image")
axes[0].axis('off')

axes[1].imshow(mask, cmap='tab10', vmin=0, vmax=7)
axes[1].set_title("Mask")
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

dummy_input = torch.randn(1, 3, 256, 256).to(device)
dummy_output = model(dummy_input)
print(f"Model: U-Net + EfficientNet-B1, Device: {device}")
print(f"Input: {dummy_input.shape} -> Output: {dummy_output.shape}")

In [ ]:
# TO DO
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks in tqdm(dataloader, desc="Training"):
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validation"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

    return running_loss / len(dataloader)

In [ ]:
# TO DO
from torch.utils.data import random_split

full_dataset = customDataset(path, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
train_losses = []
val_losses = []

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Starting training for {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Train: {train_loss:.4f} | Val: {val_loss:.4f}\n")

print("Training completed!\n")

plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs+1), train_losses, 'b-', marker='o', label='Training Loss')
plt.plot(range(1, num_epochs+1), val_losses, 'r-', marker='s', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#TO DO
def visualize_predictions(model, dataset, device, num_samples=5):
    model.eval()
    indices = np.random.choice(len(dataset), size=num_samples, replace=False)
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, mask = dataset[idx]
            image_input = image.unsqueeze(0).to(device)
            output = model(image_input)
            pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu()

            img_vis = torch.clamp(image.cpu() * std + mean, 0, 1)



            if num_samples == 1:
                ax_img, ax_gt, ax_pred = axes
            else:
                ax_img, ax_gt, ax_pred = axes[i]

            ax_img.imshow(img_vis.permute(1, 2, 0))
            ax_img.set_title('input image')
            ax_img.axis('off')
            ax_gt.imshow(mask.cpu(), cmap='tab10', vmin=0, vmax=7)
            ax_gt.set_title('ground truth')
            ax_gt.axis('off')
            ax_pred.imshow(pred_mask, cmap='tab10', vmin=0, vmax=7)
            ax_pred.set_title('prediction')
            ax_pred.axis('off')

    plt.tight_layout()
    plt.show()


visualize_predictions(model, val_dataset, device, num_samples=5)

In [ ]:
#TO DO
def calculate_pixel_accuracy(model, dataloader, device):
    model.eval()
    correct_pixels = 0
    total_pixels = 0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Calculating accuracy"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            pred_masks = torch.argmax(outputs, dim=1)
            correct_pixels += (pred_masks == masks).sum().item()
            total_pixels += masks.numel()
    return (correct_pixels / total_pixels) * 100



train_accuracy = calculate_pixel_accuracy(model, train_loader, device)
val_accuracy = calculate_pixel_accuracy(model, val_loader, device)

print(f"training pixel acc : {train_accuracy:.2f}%")
print(f"validation pixel acc: {val_accuracy:.2f}%")